# Red AI — Q4_K_M build\nThis notebook is prepared to build the model from the existing repository source.

In [ ]:
!git clone -b red-ai-build https://github.com/GiggleLootCoin/little-reds-big-studio-f36b7ec4.git red-ai-build\n%cd red-ai-build/training\n!pip -q install -U torch transformers datasets trl peft accelerate sentencepiece\n

In [ ]:
!python train_red_ai.py\n

In [ ]:
%%writefile merge_model.py\n# Merge the LoRA adapter and create a Q4_K_M GGUF.
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE="Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER="red_ai_lora"
MERGED=Path("red_ai_merged")

base=AutoModelForCausalLM.from_pretrained(BASE, torch_dtype="auto")
model=PeftModel.from_pretrained(base, ADAPTER)
model=model.merge_and_unload()
model.save_pretrained(MERGED, safe_serialization=True)
AutoTokenizer.from_pretrained(BASE).save_pretrained(MERGED)
print("merged model:", MERGED)
!python merge_model.py\n

In [ ]:
# Smoke test with llama.cpp CLI
!cmake --build /content/llama.cpp/build --config Release -j 2 --target llama-cli
!/content/llama.cpp/build/bin/llama-cli -m Red_AI_Qwen2.5-0.5B_Q4_K_M.gguf -n 80 -p 'Did you search the web for this answer?'


In [ ]:
# Smoke test\n!pip -q install llama-cpp-python\nfrom llama_cpp import Llama\nllm=Llama(model_path='Red_AI_Qwen2.5-0.5B_Q4_K_M.gguf', n_ctx=2048, verbose=False)\nout=llm.create_chat_completion(messages=[{'role':'system','content':open('system_prompt.txt').read()},{'role':'user','content':'Did you search the web for this answer?'}],max_tokens=120)\nprint(out['choices'][0]['message']['content'])\n

In [ ]:
from google.colab import files\nfiles.download('Red_AI_Qwen2.5-0.5B_Q4_K_M.gguf')\n